In [0]:
select * 
from retail.bright.brightcoffee limit 10;

--all colums are in correct format 
--changed unit price format to numbers
select
cast(replace(unit_price,',','.')as double) as unit_price
from retail.bright.brightcoffee;

--check number of records
SELECT COUNT(*)
FROM retail.bright.brightcoffee;

---check for duplicates
select*,
 count(*) 
from retail.bright.brightcoffee
group by all
having count (*) > 1;---no duplicates

---check for NULL values
SELECT *
FROM retail.bright.brightcoffee
WHERE transaction_id IS NULL
   OR transaction_date IS NULL
   OR transaction_time IS NULL
   OR transaction_qty IS NULL
   OR store_id IS NULL
   OR store_location IS NULL
   OR product_id IS NULL
   OR unit_price IS NULL
   OR product_category IS NULL
   OR product_type IS NULL
   OR product_detail IS NULL;---No Null values


-------------------------------------------------------------------------------------------1-TOTAL REVENUE
SELECT
ROUND(SUM(transaction_qty *cast(replace(unit_price,',','.')as double)),2) AS total_revenue
FROM retail.bright.brightcoffee;

--2-TOTAL REVENUE PER PRODUCT CATEGORY
SELECT PRODUCT_CATEGORY,
ROUND(SUM(transaction_qty *cast(replace(unit_price,',','.')as double)),2) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY 1;

--3-TOTAL REVENUE PER PRODUCT CATEGORY AND PRODUCT DETAILS
SELECT product_category,
product_detail,
SUM(transaction_qty *cast(replace(unit_price,',','.')as decimal(10,2))) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY ALL;

--4-TOTAL REVENUE PER PRODUCT TYPE
SELECT product_type,
SUM(transaction_qty *cast(replace(unit_price,',','.')as decimal(10,2))) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY ALL;

--5-TOTAL REVENUE PER STORE LOCATION
SELECT store_location,
ROUND(SUM(transaction_qty *cast(replace(unit_price,',','.')as double)),2) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY ALL;

--6-TOTAL REVENUE BASED ON THE TIME OF THE DAY(TIME BUCKETS)
--6am - 11:59 Morning
--12pm to 16:59> Afternoon
--17pm to 19:59> Evening

SELECT
    min(transaction_time),
    max(transaction_time)
    FROM retail.bright.brightcoffee;


SELECT product_category,
date_format(transaction_time, 'HH:mm:ss') AS transaction_time,
    CASE
WHEN date_format(transaction_time, 'HH:mm:ss') BETWEEN '06:00:00' AND '11:59:00'
            THEN 'Morning'
WHEN date_format(transaction_time, 'HH:mm:ss') BETWEEN '12:00:00' AND '16:59:00'
            THEN 'Afternoon'
WHEN date_format(transaction_time, 'HH:mm:ss') BETWEEN '17:00:00' AND '19:59:00'
            THEN 'Evening'
ELSE 'Night'
END AS transaction_time_bucket,
SUM(transaction_qty * CAST(REPLACE(unit_price, ',', '.') AS DECIMAL(10,2))) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY ALL;

--7-Date excration(month name,month id,day name)
SELECT
    transaction_date,
    MONTHNAME(transaction_date) AS month_name,
    MONTH(transaction_date) AS month_number,
    DATE_FORMAT(transaction_date, 'yyyy-MM-dd') AS month_id,
    DAYNAME(transaction_date) AS day_name,
    DAYOFWEEK(transaction_date) AS day_number
FROM retail.bright.brightcoffee;

---TOTAL REVENUE PER DAY
SELECT
    transaction_date,
    MONTHNAME(transaction_date) AS month_name,
    COUNT(transaction_id) AS transaction_count
FROM retail.bright.brightcoffee
GROUP BY ALL
ORDER BY transaction_date;
-----------------------------------------------------------------------------------------
SELECT
    transaction_date,
    MONTHNAME(transaction_date) AS month_name,
    MONTH(transaction_date) AS month_number,
    DATE_FORMAT(transaction_date, 'yyyy-MM-dd') AS month_id,
    DAYNAME(transaction_date) AS day_name,
    DAYOFWEEK(transaction_date) AS day_number,
    COUNT(transaction_id) AS transaction_count,
    SUM(transaction_qty) AS products_sold,
    product_category,
    product_detail,
    product_type,
    store_location,
CASE
WHEN date_format(transaction_time, 'HH:mm:ss')
BETWEEN '06:00:00' AND '11:59:00'
THEN 'Morning'
WHEN date_format(transaction_time, 'HH:mm:ss')
BETWEEN '12:00:00' AND '16:59:00'
THEN 'Afternoon'
WHEN date_format(transaction_time, 'HH:mm:ss')
BETWEEN '17:00:00' AND '19:59:00'
THEN 'Evening'
ELSE 'Night'
END AS transaction_time_bucket,
ROUND(SUM(transaction_qty *CAST(REPLACE(unit_price, ',', '.') AS DOUBLE)),2) AS total_revenue
FROM retail.bright.brightcoffee
GROUP BY ALL
ORDER BY transaction_date;


SELECT *
FROM retail.bright.brightcoffee_silver
LIMIT 10;
DESCRIBE retail.bright.brightcoffee_silver;

CREATE OR REPLACE TABLE retail.bright.brightcoffee_gold AS

SELECT
    transaction_id,
    transaction_date,
    transaction_time,
    transaction_qty,
    store_id,
    store_location,
    product_id,
    unit_price_clean AS unit_price,
    product_category,
    product_type,
    product_detail,
    total_amount,
    transaction_time_bucket,

    MONTHNAME(transaction_date) AS month_name,
    MONTH(transaction_date) AS month_number,
    DAYNAME(transaction_date) AS day_name,
    DAYOFWEEK(transaction_date) AS day_number

FROM retail.bright.brightcoffee_silver;
